In [2]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

In [3]:
DB_USER = "postgres"
DB_PASSWORD = "your_password"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "customer_revenue_platform"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connected Successfully")

Connected Successfully


In [4]:
query = """
SELECT *
FROM customer_features
"""

customer_df = pd.read_sql(
    query,
    engine
)

In [14]:
query = """
SELECT *
FROM xgboost_predictions
"""

pred_df = pd.read_sql(
    query,
    engine
)

In [15]:
query = """
SELECT *
FROM customer_segments
"""

segment_df = pd.read_sql(
    query,
    engine
)

In [16]:
revenue_df = customer_df.merge(
    pred_df[
        [
            "customer_unique_id",
            "purchase_probability"
        ]
    ],
    on="customer_unique_id",
    how="left"
)

revenue_df = revenue_df.merge(
    segment_df[
        [
            "customer_unique_id",
            "customer_segment"
        ]
    ],
    on="customer_unique_id",
    how="left"
)

In [17]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

revenue_df[
    [
        "revenue_score",
        "value_score",
        "recency_score"
    ]
] = scaler.fit_transform(
    revenue_df[
        [
            "total_revenue",
            "customer_value_score",
            "recency_days"
        ]
    ]
)

In [18]:
revenue_df["recency_score"] = (
    1 - revenue_df["recency_score"]
)

In [25]:
PURCHASE_WEIGHT = 0.30
VALUE_WEIGHT = 0.30
REVENUE_WEIGHT = 0.20
RECENCY_WEIGHT = 0.20

revenue_df["revenue_opportunity_score"] = (
    revenue_df["purchase_probability"] * PURCHASE_WEIGHT
    +
    revenue_df["value_score"] * VALUE_WEIGHT
    +
    revenue_df["revenue_score"] * REVENUE_WEIGHT
    +
    revenue_df["recency_score"] * RECENCY_WEIGHT
) * 100

In [20]:
revenue_df["opportunity_level"] = pd.cut(
    revenue_df["revenue_opportunity_score"],
    bins=[0, 40, 70, 100],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

In [21]:
revenue_df[
    [
        "customer_unique_id",
        "revenue_opportunity_score",
        "opportunity_level"
    ]
].sort_values(
    by="revenue_opportunity_score",
    ascending=False
).head(20)

,customer_unique_id,revenue_opportunity_score,opportunity_level
75269,c8460e4251689ba205045f3ea17884a1,83.281420,High
96083,fff5eb4918b2bf4b2da476788d42051c,80.187306,High
26456,46450c74a0d8c5ca9395da1daac6c120,80.181237,High
14363,262e1f1e26e92e86375f86840b4ffd63,79.793418,High
54182,906a8a4ec9f3d4c3e64fa6d1c4fe6009,79.705317,High
65954,af5454198a97379394cacf676e1e96cb,79.603234,High
85697,e3fe811011101628e80a7953f1244c8d,79.539812,High
58710,9c3af16efacb7aa06aa3bc674556c5d6,79.493740,High
70105,ba84da8c159659f116329563a0a981dd,79.312344,High
84230,e015ce18751465bc79eeabbe3f0064d5,79.267027,High


In [22]:
revenue_df.to_sql(
    "revenue_opportunity_scores",
    con=engine,
    if_exists="replace",
    index=False
)

96

In [23]:
summary = (
    revenue_df
    .groupby("opportunity_level")
    .agg({
        "customer_unique_id":"count",
        "total_revenue":"mean",
        "purchase_probability":"mean",
        "revenue_opportunity_score":"mean"
    })
    .reset_index()
)

summary

,opportunity_level,customer_unique_id,total_revenue,purchase_probability,revenue_opportunity_score
0,Low,2594,54.720428,0.995774,37.702125
1,Medium,85188,194.978166,0.996530,56.575325
2,High,8314,460.418475,0.997225,73.054153


In [24]:
summary.to_sql(
    "revenue_opportunity_summary",
    con=engine,
    if_exists="replace",
    index=False
)

3